<a href="https://colab.research.google.com/github/Aditi-Chopra/ju-cmater-project/blob/main/Data_Pre_processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install tensorflow statsmodels scikeras shap

In [ ]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import VarianceThreshold

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Consolidated Data.csv')

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Colab Notebooks/Consolidated Data.csv'

In [ ]:
# Step 1: Defining Target / y-variable: risk
target_col = 'risk'

if target_col in df.columns:
    y = df[target_col].copy()
    X = df.drop(columns=[target_col]).copy()
    print(f"Step 1: Target '{target_col}' isolated. Features shape: {X.shape}")

else:
    raise KeyError(f"Target column '{target_col}' not found.")

In [ ]:
# Step 2: Drop Identifiers and Leakages
cols_to_drop = [
    'event_id', 'mission_id',
    'max_risk_estimate', 'max_risk_scaling',
    't_time_lastob_start', 't_time_lastob_end',
    'c_time_lastob_start', 'c_time_lastob_end',
]

dropped_cols = [col for col in cols_to_drop if col in X.columns]
X.drop(columns=dropped_cols, inplace=True)

In [ ]:
# Step 3A: Clean Missing Data (Imputation):
# Identifying which columns are text/categories(ordinal) and which are numbers

cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

num_imputer = SimpleImputer(strategy='median')
if num_cols:
    X[num_cols] = num_imputer.fit_transform(X[num_cols])

cat_imputer = SimpleImputer(strategy='most_frequent')
if cat_cols:
    X[cat_cols] = cat_imputer.fit_transform(X[cat_cols])

In [ ]:
# Step 3B: Encode Categories and Scale Numbers
# 1. Defining the simultaeneous processing rules
preprocessor = ColumnTransformer(
    transformers= [
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False),
         cat_cols)
    ],
    remainder='passthrough',
    verbose_feature_names_out=False
)

# 2. Applying the transformation to the feature set
X_processed = preprocessor.fit_transform(X)

In [ ]:
# Step 4:
# 1. Convert the preprocessed array into a Pandas DataFrame with feature names
feature_names = preprocessor.get_feature_names_out()

X_scaled = pd.DataFrame(X_processed, columns=feature_names)

# 2. Define variance threshold
variance_threshold = 0.01

# Option A: Direct Pandas approach (Efficient equivalent to np.diag(cov_matrix))
feature_variances = X_scaled.var()

nzv_cols = feature_variances[feature_variances < variance_threshold].index.tolist()

X_final = X_scaled.drop(columns=nzv_cols)

print(f"Features removed due to low variance (< {variance_threshold}): {len(nzv_cols)}")
print(f"Names of removed features: {nzv_cols}")
print(f"Final feature matrix shape: {X_final.shape}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
import scipy.cluster.hierarchy as sch
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.regularizers import l1
from tensorflow.keras.callbacks import EarlyStopping
from scikeras.wrappers import KerasClassifier
import shap
import warnings
warnings.filterwarnings('ignore')

X_df = X_final.copy()
# copies the preprocessed feature matrix to prevent overwriting original data

In [ ]:
# Step 5: Correlation, Dendrogram, and Multicollinearity Analysis

# 1. Computing correlation matrix and p-values
def compute_correlation_with_pvalues(dataframe):
  cols = dataframe.columns
  corr_matrix = pd.DataFrame(index=cols, columns=cols, dtype=float)
  p_matrix = pd.DataFrame(index=cols, columns=cols, dtype=float)

  for col1 in cols:
    for col2 in cols:
      r, p = pearsonr(dataframe[col1], dataframe[col2])
      corr_matrix.loc[col1, col2] = r
      p_matrix.loc[col1, col2] = p

  return corr_matrix, p_matrix

corr_matrix, p_matrix = compute_correlation_with_pvalues(X_df)

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', vmin=-1, vmax=1)
plt.title("Feature Correlation Matrix")
plt.tight_layout()
plt.savefig("correlation_heatmap.jpeg")
plt.show()

# # 3. Plotting Feature Distributions acroos Target
# plot_df = X_df.copy()
# plot_df[target_col] = y.values

# for col in X_df.columns[:10]:
# # features for trend visualisation
#   plt.figure(figsize=(10, 8))
#   sns.kdeplot(data=plot_df, x=col, hue=target_col, common_norm=False, palette='Set2')
#   plt.title("Trend & Distribution of {col} across {target_col}")
#   plt.tight_layout()  # adjusts padding
#   plt.show()

# # 4. Hirerachical Clustering Dendogram
# distance_matrix = 1 - np.abs(corr_matrix)
# linkage = sch.linkage(sch.distance.squareform(distance_matrix), method='ward')

# plt.figure(figsize=(10, 6))
# dendogram = sch.dendogram(linkage, labels=X_df.columns, orientation='top', leaf_rotation=90)
# plt.title("Hierarchical Feature Clustering Dendrogram")
# plt.ylabel("Distance (1 - |Correlation|)")
# plt.tight_layout()
# plt.savefig("feature_dendrogram.jpeg")
# plt.show()

In [ ]:
# 5. Removing Redundant Features
high_corr_threshold = 0.8
alpha_sig = 0.05

upper_tri_corr = corr_matrix.abs().where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
upper_tri_p = p_matrix.where(np.triu(np.ones(p_matrix.shape), k=1).astype(bool))

to_drop_corr = []
for col in upper_tri_corr.columns:
    if any((upper_tri_corr[col] > high_corr_threshold) & (upper_tri_p[col] < alpha_sig)):
        to_drop_corr.append(col)

print(f"Features dropped due to significant high correlation: {to_drop_corr}")
X_df_reduced = X_df.drop(columns=to_drop_corr)

# 6. Variance Inflation Factor (VIF) Screening
def calculate_vif(dataframe):
    vif_data = pd.DataFrame()
    vif_data["Feature"] = dataframe.columns
    vif_data["VIF"] = [variance_inflation_factor(dataframe.values, i) for i in range(dataframe.shape[1])]
    return vif_data

vif_threshold = 10.0
vif_df = calculate_vif(X_df_reduced)

to_drop_vif = vif_df[vif_df['VIF'] > vif_threshold]['Feature'].tolist()

print(f"Features dropped due to high VIF (> {vif_threshold}): {to_drop_vif}")

X_step5 = X_df_reduced.drop(columns=to_drop_vif)
final_feature_names = X_step5.columns.tolist()

In [ ]:
# Step 6: Reshaping for LSTM & Define Base Model Architecture

# 1. Transform the 2D feature DataFrame (X_step5) into a 3D tensor for the LSTM:
# (samples, timesteps, features)
X_lstm = X_step5.values.reshape((X_step5.shape[0], 1, X_step5.shape[1]))
print(f"Successfully reshaped X_step5 into X_lstm with shape: {X_lstm.shape}")

# 2. Force y to match the exact row count of X_lstm (Foolproof alignment)
y_array = y.values if hasattr(y, 'values') else np.array(y)
if len(y_array) > X_lstm.shape[0]:
    y_aligned = y_array[:X_lstm.shape[0]]
else:
    y_aligned = y_array

print(f"Aligned shapes -> X_lstm: {X_lstm.shape[0]} rows | y_aligned: {len(y_aligned)} rows")

# 3. Split data into training and testing sets (Clean regression split, no stratification)
X_train, X_test, y_train, y_test = train_test_split(
    X_lstm,
    y_aligned,
    test_size=0.2,
    random_state=42
)

print(f"\nTraining feature shape: {X_train.shape}, Testing feature shape: {X_test.shape}")
print(f"Training target shape: {y_train.shape}, Testing target shape: {y_test.shape}")

# 4. Define the LSTM regression model architecture
def create_lstm_model(input_features):
    model = Sequential()

    model.add(LSTM(32, input_shape=(1, input_features), kernel_regularizer=l1(0.01), return_sequences=False))

    model.add(Dropout(0.2))
    model.add(Dense(16, activation='relu'))

    # Regression output layer: 1 neuron with linear activation to predict continuous risk scores
    model.add(Dense(1, activation='linear'))

    model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])

    return model

In [ ]:
# Step 7: Embedded Feature Selection

import tensorflow as tf
from tensorflow.keras import backend as K

# 1. Clearing the TensorFlow session and global gradient registry
K.clear_session()

# 2. Re-initialising the base LSTM model cleanly
base_model = create_lstm_model(input_features=X_train.shape[2])

# 3. Configuring early stopping to prevent overtraining
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# 4. Training the base model
base_model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=0
)

print("Embedded feature selection model trained successfully!")

In [ ]:
# STEP 8: SHAP Analysis

from sklearn.ensemble import RandomForestRegressor

# Flattening 3D arrays back to 2D for the tree proxy
X_train_2d = X_train[:, 0, :]
X_test_2d = X_test[:, 0, :]

# Training a Random Forest Regressor proxy model
proxy_model = RandomForestRegressor(n_estimators=100, random_state=42)
proxy_model.fit(X_train_2d, y_train)

# Initialising SHAP TreeExplainer for regression
explainer = shap.TreeExplainer(proxy_model)
shap_values = explainer.shap_values(X_test_2d)

# Ensuring 2D shape for Pandas compatibility
shap_vals_arr = shap_values[0] if isinstance(shap_values, list) else shap_values
if shap_vals_arr.ndim == 3:
    shap_vals_arr = shap_vals_arr[:, :, 0]

# Calculating mean absolute SHAP values for global feature ranking
mean_shap_values = np.abs(shap_vals_arr).mean(axis=0).flatten()
shap_importance_df = pd.DataFrame({
    'Feature': final_feature_names,
    'SHAP_Importance': mean_shap_values
}).sort_values(by='SHAP_Importance', ascending=False)

ranked_features = shap_importance_df['Feature'].tolist()
print("Features ranked by SHAP Importance:\n", shap_importance_df)

# Rendering SHAP Summary Plot
shap.summary_plot(shap_vals_arr, X_test_2d, feature_names=final_feature_names, show=False)
plt.title("SHAP Global Feature Importance (Regression)")
plt.savefig("SHAP_summary.png", bbox_inches='tight')
plt.show()

In [ ]:
# Step 9: Incremental Feature Addition & Model Significance

from sklearn.metrics import mean_absolute_error

best_mae = float('inf')
selected_features = []

for feature in ranked_features:
    test_features = selected_features + [feature]

    feat_indices = [final_feature_names.index(f) for f in test_features]
    X_train_sub = X_train[:, :, feat_indices]
    X_test_sub = X_test[:, :, feat_indices]

    # Training temporary regression LSTM
    temp_model = create_lstm_model(input_features=len(test_features))
    temp_model.fit(X_train_sub, y_train, epochs=15, batch_size=32, verbose=0)

    # Predicting and calculate Mean Absolute Error (MAE)
    y_pred = temp_model.predict(X_test_sub, verbose=0).flatten()
    current_mae = mean_absolute_error(y_test, y_pred)

    # MAE: Mean Absolute Error -> loss function used for regression models
    if current_mae <= (best_mae + 0.005):
        selected_features.append(feature)
        best_mae = current_mae
        print(f"Added '{feature}' | New Test MAE: {best_mae:.4f}")
    else:
        print(f"Rejected '{feature}' | Test MAE increased to {current_mae:.4f}")

print(f"\nFinal Selected Features from Incremental Addition: {selected_features}")

In [ ]:
# Step 10: Final Feature Subset & Final Model
final_feature_indices = [final_feature_names.index(f) for f in selected_features]

X_train_final = X_train[:, :, final_feature_indices]
X_test_final = X_test[:, :, final_feature_indices]

final_model = create_lstm_model(input_features=len(selected_features))

final_history = final_model.fit(
    X_train_final, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

y_pred_final_prob = final_model.predict(X_test_final)
y_pred_final = (y_pred_final_prob > 0.5).astype(int)

print(classification_report(y_test, y_pred_final))

final_model.save('Risk_Prediction_LSTM_Final.h5')